# Standard Calendar Table — caresyncdata.silver.calendar

Generates a full date-dimension table from **1940-01-10 to 2030-12-31** covering all standard calendar attributes: date keys, year, quarter, month, week, day, ISO fields, period boundaries, and flags.

In [0]:
from pyspark.sql import functions as F

# ── 1. Generate every date in range ───────────────────────────────────────────
date_range = spark.sql("""
    SELECT explode(
        sequence(DATE('1940-01-10'), DATE('2030-12-31'), INTERVAL 1 DAY)
    ) AS full_date
""")

# ── 2. Build all calendar columns ─────────────────────────────────────────────
calendar_df = date_range.select(

    # ── Keys ──────────────────────────────────────────────────────────────────
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("full_date"),

    # ── Year ──────────────────────────────────────────────────────────────────
    F.year("full_date").alias("year"),

    # ── Half-Year ─────────────────────────────────────────────────────────────
    F.when(F.month("full_date") <= 6, 1).otherwise(2).alias("half_year"),
    F.when(F.month("full_date") <= 6, "H1").otherwise("H2").alias("half_year_name"),

    # ── Quarter ───────────────────────────────────────────────────────────────
    F.quarter("full_date").alias("quarter"),
    F.concat(F.lit("Q"), F.quarter("full_date")).alias("quarter_name"),
    F.concat(
        F.year("full_date").cast("string"), F.lit("-Q"), F.quarter("full_date")
    ).alias("year_quarter"),

    # ── Month ─────────────────────────────────────────────────────────────────
    F.month("full_date").alias("month_number"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.date_format("full_date", "MMM").alias("month_name_short"),
    F.date_format("full_date", "yyyy-MM").alias("year_month"),           # sortable
    F.date_format("full_date", "MMM-yyyy").alias("month_year_label"),    # display

    # ── Week ──────────────────────────────────────────────────────────────────
    F.weekofyear("full_date").alias("week_of_year"),                     # ISO week 1-53
    F.ceil(F.dayofmonth("full_date") / 7).cast("int").alias("week_of_month"),
    F.trunc("full_date", "week").alias("start_of_week"),                 # Monday (ISO)
    F.expr("date_add(trunc(full_date, 'week'), 6)").alias("end_of_week"),# Sunday

    # ── Day ───────────────────────────────────────────────────────────────────
    F.dayofmonth("full_date").alias("day_of_month"),
    F.dayofyear("full_date").alias("day_of_year"),

    # Spark: 1=Sun … 7=Sat  |  ISO: 1=Mon … 7=Sun
    F.dayofweek("full_date").alias("day_of_week"),
    ((F.dayofweek("full_date") + 5) % 7 + 1).alias("day_of_week_iso"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.date_format("full_date", "EEE").alias("day_name_short"),

    # ── Flags ─────────────────────────────────────────────────────────────────
    F.when(F.dayofweek("full_date").isin(1, 7), True)
     .otherwise(False).alias("is_weekend"),
    F.when(F.dayofweek("full_date").isin(1, 7), False)
     .otherwise(True).alias("is_weekday"),
    F.when(
        (F.year("full_date") % 4 == 0) &
        ((F.year("full_date") % 100 != 0) | (F.year("full_date") % 400 == 0)),
        True
    ).otherwise(False).alias("is_leap_year"),

    # ── Period Boundaries ─────────────────────────────────────────────────────
    F.trunc("full_date", "month").alias("start_of_month"),
    F.last_day("full_date").alias("end_of_month"),
    F.trunc("full_date", "quarter").alias("start_of_quarter"),
    F.expr("last_day(add_months(trunc(full_date,'quarter'), 2))").alias("end_of_quarter"),
    F.trunc("full_date", "year").alias("start_of_year"),
    F.expr("make_date(year(full_date), 12, 31)").alias("end_of_year"),

    # ── Helpers ───────────────────────────────────────────────────────────────
    F.dayofmonth(F.last_day("full_date")).alias("days_in_month"),
    F.datediff(
        F.last_day("full_date"),
        F.trunc("full_date", "month")
    ).alias("days_remaining_in_month"),
)

# ── 3. Write to Delta ──────────────────────────────────────────────────────────
(
    calendar_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("caresyncdata.silver.calendar")
)

row_count = spark.table("caresyncdata.silver.calendar").count()
print(f"\u2705 caresyncdata.silver.calendar created successfully")
print(f"   Rows  : {row_count:,}")
print(f"   Range : 1940-01-10  →  2030-12-31")
print(f"   Cols  : {len(calendar_df.columns)}")
